# Cómo elige palabras un LLM: temperatura, top-k y top-p

**Facsímil 3 · Arquitecturas y modelos** — capítulo 4 (MLP, residual, *logits* y *sampling*).

Aquí desmontamos una de las confusiones más comunes sobre los LLM: la idea de que el modelo «elige» la
siguiente palabra. No la elige. El modelo produce una **puntuación** (un *logit*) para *cada* palabra de
su vocabulario, y un proceso de **muestreo** —que tú controlas con tres botones— decide cuál sale. En
este cuaderno te pones en la piel de esa última capa y ves cómo la **temperatura**, el **top-k** y el
**top-p** transforman al mismo modelo de un loro predecible a un poeta disparatado. Entender estos
botones es dejar de sufrir respuestas «raras» y empezar a controlarlas.

### Qué vas a aprender
- Qué son los **logits** y cómo la **softmax** los convierte en una distribución de probabilidad.
- Cómo la **temperatura** modela el atrevimiento del modelo, con su fórmula y su intuición.
- Qué hacen **top-k** y **top-p** (núcleo) y en qué se diferencian.
- A medir la **incertidumbre** de la distribución (entropía) y a *ver* la diferencia entre estrategias
  muestreando muchas veces.

### Cuánto cuesta
Unos 12 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
import numpy as np
np.random.seed(0)

# Despues de "El cafe de la manana me ayuda a ___", estas son las puntuaciones
# (logits) que daria el modelo a algunas continuaciones. Las ponemos a mano para
# ver la mecanica; en un LLM real salen de la red (capitulos 2-4).
vocab  = ["despertar", "concentrarme", "trabajar", "vivir", "pensar", "volar", "programar", "bailar"]
logits = np.array([3.2, 2.8, 2.5, 1.4, 1.2, -0.5, 0.8, -1.0])

def softmax(z):
    z = z - z.max(); e = np.exp(z); return e / e.sum()

p = softmax(logits)
print("Distribucion 'cruda' (softmax de los logits):")
for w, pi in sorted(zip(vocab, p), key=lambda t: -t[1]):
    print(f"  {w:>14}: {pi*100:5.1f}%  {'#'*int(pi*40)}")


Distribucion 'cruda' (softmax de los logits):
       despertar:  38.5%  ###############
    concentrarme:  25.8%  ##########
        trabajar:  19.1%  #######
           vivir:   6.4%  ##
          pensar:   5.2%  ##
       programar:   3.5%  #
           volar:   1.0%  
          bailar:   0.6%  


## 1. De logits a probabilidades: la softmax

La última capa del modelo no da probabilidades, da **logits**: números reales sin acotar, uno por cada
palabra del vocabulario (que en un LLM real tiene decenas de miles de entradas). Para convertirlos en
una distribución de probabilidad —números entre 0 y 1 que suman 1— se aplica la **softmax**:
$$ p_i = \frac{e^{z_i}}{\sum_j e^{z_j}} $$
La exponencial agranda las diferencias (lo más probable se separa de lo improbable) y la división
normaliza. El resultado es la distribución que ves arriba: el modelo «cree» que «despertar» es lo más
probable, pero deja una rendija a las demás. Lo que viene ahora es **moldear** esa distribución.


## 2. La temperatura: subir o bajar el atrevimiento

La temperatura $T$ divide los *logits* antes de la softmax: $p_i \propto e^{z_i / T}$. La intuición
viene de la física (de ahí el nombre): a temperatura **baja**, el sistema se «congela» en su estado más
probable; a temperatura **alta**, se agita y visita estados improbables.

- $T < 1$ (p. ej. 0.5): **agranda** las diferencias. El modelo se vuelve seguro, predecible,
  repetitivo. Ideal para extraer datos o clasificar.
- $T = 1$: la distribución original.
- $T > 1$ (p. ej. 1.5): **aplana** las diferencias. Da oportunidades a palabras improbables: aparece la
  creatividad... y los disparates.


In [2]:
print(f"{'palabra':>14} | T=0.5    T=1.0    T=1.5")
for i, w in enumerate(vocab):
    fila = "   ".join(f"{softmax(logits/T)[i]*100:4.0f}%" for T in [0.5, 1.0, 1.5])
    print(f"{w:>14} |  {fila}")
print("\nT baja concentra en 'despertar'; T alta reparte y da aire incluso a 'volar' o 'bailar'.")


       palabra | T=0.5    T=1.0    T=1.5
     despertar |    57%     38%     30%
  concentrarme |    26%     26%     23%
      trabajar |    14%     19%     19%
         vivir |     2%      6%      9%
        pensar |     1%      5%      8%
         volar |     0%      1%      3%
     programar |     0%      3%      6%
        bailar |     0%      1%      2%

T baja concentra en 'despertar'; T alta reparte y da aire incluso a 'volar' o 'bailar'.


## 3. Medir la incertidumbre: la entropía

¿Cómo ponemos número a «cómo de repartida» está una distribución? Con la **entropía**: $H = -\sum_i p_i
\log_2 p_i$. Vale 0 cuando el modelo está completamente seguro (toda la probabilidad en una palabra) y
crece cuanto más repartida está (máxima cuando todas son igual de probables). La temperatura es, en el
fondo, un mando de entropía: subirla aumenta la incertidumbre de la siguiente palabra.


In [3]:
def entropia(p):
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

print("temperatura | entropia (bits) | interpretacion")
for T in [0.2, 0.5, 1.0, 1.5, 3.0]:
    h = entropia(softmax(logits/T))
    print(f"    {T:>4}    |     {h:4.2f}      | {'casi decidido' if h<1 else 'algo de variedad' if h<2 else 'muy abierto'}")
print(f"\nMaxima entropia posible con {len(vocab)} palabras: {np.log2(len(vocab)):.2f} bits (todas iguales).")


temperatura | entropia (bits) | interpretacion
     0.2    |     0.69      | casi decidido
     0.5    |     1.57      | algo de variedad
     1.0    |     2.24      | muy abierto
     1.5    |     2.56      | muy abierto
     3.0    |     2.86      | muy abierto

Maxima entropia posible con 8 palabras: 3.00 bits (todas iguales).


## 4. Top-k y top-p: cortar la cola de los disparates

La temperatura cambia la *forma* de toda la distribución. Otra familia de técnicas, en cambio,
**recorta**: deja fuera las opciones malas para que nunca se muestreen, por improbables que sean tras
subir la temperatura.

- **Top-k:** se queda solo con las `k` palabras más probables y reparte entre ellas. Simple, pero `k`
  es fijo: a veces corta de más, a veces de menos.
- **Top-p (núcleo):** se queda con las palabras que **acumulan** el `p`% de probabilidad. Es
  adaptativo: cuando el modelo lo tiene claro, mantiene pocas; cuando duda, mantiene muchas. Suele dar
  mejores resultados que top-k.


In [4]:
def top_k(p, k):
    q = p.copy(); q[np.argsort(p)[:-k]] = 0; return q / q.sum()
def top_p(p, umbral):
    orden = np.argsort(p)[::-1]; acum = np.cumsum(p[orden])
    mantener = orden[:np.searchsorted(acum, umbral) + 1]
    q = np.zeros_like(p); q[mantener] = p[mantener]; return q / q.sum()

print("Top-k=3 (solo las 3 mejores; el resto, prohibido):")
for w, pi in sorted(zip(vocab, top_k(p, 3)), key=lambda t:-t[1]):
    if pi > 0: print(f"  {w:>14}: {pi*100:5.1f}%")
print("\nTop-p=0.9 (las que juntas suman el 90%):")
mantenidas = sum(1 for pi in top_p(p, 0.9) if pi > 0)
for w, pi in sorted(zip(vocab, top_p(p, 0.9)), key=lambda t:-t[1]):
    if pi > 0: print(f"  {w:>14}: {pi*100:5.1f}%")
print(f"  -> top-p mantuvo {mantenidas} palabras (las justas para sumar el 90%).")


Top-k=3 (solo las 3 mejores; el resto, prohibido):
       despertar:  46.1%
    concentrarme:  30.9%
        trabajar:  22.9%

Top-p=0.9 (las que juntas suman el 90%):
       despertar:  40.5%
    concentrarme:  27.2%
        trabajar:  20.1%
           vivir:   6.7%
          pensar:   5.5%
  -> top-p mantuvo 5 palabras (las justas para sumar el 90%).


## 5. Lo que de verdad cambia: generar muchas veces

Los porcentajes son abstractos. Hagámoslo tangible: muestreamos 2000 veces la siguiente palabra con
cada estrategia y contamos. Así *ves* la diferencia entre «siempre lo más probable» (*greedy*, que es
el límite $T \to 0$), «con algo de variedad» y «barra libre».


In [5]:
def muestrea(p, n=2000):
    idx = np.random.choice(len(p), size=n, p=p)
    cuenta = {vocab[i]: int((idx==i).sum()) for i in range(len(p)) if (idx==i).sum() > 0}
    return dict(sorted(cuenta.items(), key=lambda t: -t[1]))

print("Greedy (T->0): SIEMPRE 'despertar' (la mas probable, sin variedad).")
print("T=1.0    :", muestrea(p))
print("T=1.5    :", muestrea(softmax(logits/1.5)))
print("Top-p 0.9:", muestrea(top_p(p, 0.9)))


Greedy (T->0): SIEMPRE 'despertar' (la mas probable, sin variedad).
T=1.0    : {'despertar': 782, 'concentrarme': 484, 'trabajar': 372, 'vivir': 127, 'pensar': 112, 'programar': 85, 'volar': 25, 'bailar': 13}
T=1.5    : {'despertar': 611, 'concentrarme': 480, 'trabajar': 367, 'vivir': 186, 'pensar': 144, 'programar': 132, 'volar': 48, 'bailar': 32}
Top-p 0.9: {'despertar': 813, 'concentrarme': 567, 'trabajar': 401, 'vivir': 134, 'pensar': 85}


**La lección.** Con *greedy* o temperatura baja, el modelo dice casi siempre lo mismo: fiable para
extraer datos, clasificar o seguir instrucciones al pie de la letra. Con temperatura alta o *top-p*
amplio, explora: bueno para escribir, malo para precisión. No hay un valor «correcto»: depende de si
quieres un **notario** o un **poeta**. Y ese botón es tuyo, no del modelo. Muchas «alucinaciones
creativas» y muchas «respuestas planas» se explican (y se domestican) justo aquí.


## 6. Pruébalo tú

1. **Pon `T = 0.1`**: ¿queda alguna palabra con opciones además de «despertar»? Eso es casi *greedy*.
2. **Combina técnicas:** aplica *top-p* y luego muestrea con `T=1.2`. Recortar la cola **y** dar
   variedad es lo que usan de serie muchos asistentes (primero filtran, luego muestrean).
3. **Empata dos opciones:** cambia los *logits* para que «trabajar» y «despertar» estén casi iguales.
   ¿Cómo afecta eso a *top-p* (que se adapta) frente a *top-k* (que es fijo)?
4. **Dibuja la entropía** frente a la temperatura (un bucle + `matplotlib`): verás la curva del
   «atrevimiento» del modelo.


## 7. Errores comunes

- **Creer que el modelo «decide» la palabra.** Da una distribución; el muestreo decide. Cambiar los
  parámetros de muestreo cambia la salida sin tocar el modelo.
- **Usar temperatura alta para tareas de precisión** (extraer un dato, clasificar). Ahí quieres
  *greedy* o `T` baja: fiabilidad, no creatividad.
- **Confundir top-k y top-p.** `k` es fijo; `p` se adapta a la confianza del modelo. En la práctica,
  top-p suele comportarse mejor.
- **Olvidar que estos botones interactúan.** Temperatura, top-k y top-p se combinan; el efecto final es
  de los tres juntos, no de uno solo.


## 8. Qué te llevas

- Un LLM produce **una distribución sobre todo el vocabulario**; el texto sale de **muestrear** esa
  distribución, no de una verdad única.
- La **temperatura** moldea la forma (atrevimiento, medible como entropía); **top-k/top-p** recortan la
  cola (seguridad). Juntos controlan el equilibrio entre fiabilidad y creatividad.
- Elegir bien estos parámetros es parte del oficio: notario para extraer y clasificar, poeta para
  escribir. La decisión es tuya.

**Para seguir:** el facsímil 4 (caja de herramientas) usa estos parámetros desde la API real; y el
facsímil 7 mide cuánto te puedes fiar de lo que sale, calibración incluida.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*